# Project 10: Baum-Welch Algorithm
This notebook implements the Baum-Welch Algorithm. It does via two main methods of the overarching class:

1. Expectation - gets the expected log soft counts for initial, transition, and emission probabilities. Relies on inheritance of our previous ForwardBackward algorithm to compute and use posterior marginal probabilities.

2. Maximization - normalizes expected counts to log probabilities based on the expected counts for a given state in each of the sets of probabilities. For initial probabilities, it normalizes the values by including counts for all states, for transtion probabilities it normalizes the expected counts for transitions to every other state, and for emissions it normalizes the expected counts for every emission in a given state.

## Main Algorithm
Print statements are included in expectation step to demonstrate the change in log-likelihood as the model is trained to demonstrate how the models perform compared to one another after each iteration. Depending on how many iterations you choose to do, you may want to remove this statement or at least change it's frequency.

In [37]:

from collections import defaultdict
from scipy.special import logsumexp
from forward_backward import *
import copy


class BaumWelch(ForwardBackward):
    """
    Implementation of the Baum-Welch algorithm
    """

    def expectation(self, sequences):
        """
        Performs expectation step of the algorithm
        sequences:
        return: log_transitions, log_emissions, log_initial
        """

        # initialize the model parameters to update for each iteration
        next_transition_counts = [[float('-inf') for _ in self.states] for _ in self.states]
        next_emission_counts = [[float('-inf') for _ in self.emissions] for _ in self.states]
        next_initial_counts = [float('-inf') for _ in self.states]


        # pass over each sequence and update the parameters
        for sequence in sequences:
            # get the probabilities needed to update model parameters based on the sequence
            forward_matrix, forward_prob = self.forward(sequence)
            backward_matrix, backward_prob = self.backward(sequence)

            # update our initial state soft counts
            for i, state in enumerate(self.states):
                log_emission = forward_matrix[state][0] + backward_matrix[state][0] - forward_prob
                next_initial_counts[i] = np.logaddexp(next_initial_counts[i], log_emission)

            # update our emission and transition counts
            for i in range(len(sequence) - 1):
                nucleotide = sequence[i]

                for j, state in enumerate(self.states):
                    log_fk = forward_matrix[state][i]
                    log_bk = backward_matrix[state][i]

                    # get emission soft counts for each state
                    log_emission = log_fk + log_bk - forward_prob
                    nucleotide_index = self.emissions.index(nucleotide)
                    next_emission_counts[j][nucleotide_index] = np.logaddexp(next_emission_counts[j][nucleotide_index], log_emission)

                    # get transition soft counts for each state
                    for k,next_state in enumerate(self.states):
                        log_akl = self.log_transition[state][next_state]
                        log_el = self.log_emission[next_state][sequence[i + 1]]
                        log_bl = backward_matrix[next_state][i + 1]

                        # update soft counts for each states transition probs
                        log_transition = log_fk + log_akl + log_el + log_bl - forward_prob
                        next_transition_counts[j][k] = np.logaddexp(next_transition_counts[j][k], log_transition)

        return next_transition_counts, next_emission_counts, next_initial_counts


    @staticmethod
    def convert_counts(transition_counts, emission_counts, initial_counts):
        """
        Converts soft counts to log probabilities
        """

        log_transition_probs = []
        for state in transition_counts:
            log_transitions = []
            for log_count in state:
                log_prob = log_count - logsumexp(state)
                log_transitions.append(log_prob)
            log_transition_probs.append(log_transitions)

        log_emission_probs = []
        for state in emission_counts:
            log_emissions = []
            for log_count in state:
                log_prob = log_count - logsumexp(state)
                log_emissions.append(log_prob)
            log_emission_probs.append(log_emissions)

        log_initial_probs = []
        for log_count in initial_counts:
            log_prob = log_count - logsumexp(initial_counts)
            log_initial_probs.append(log_prob)


        return log_transition_probs, log_emission_probs, log_initial_probs


    def maximization(self, sequences, threshold, iterations):
        """
        Performs maximization step of the Baum-Welch algorithm
        """

        for iter_idx in range(iterations):
            # make a copy of our current model
            transition_copy = copy.deepcopy(self.log_transition)
            emission_copy = copy.deepcopy(self.log_emission)
            initial_copy = copy.deepcopy(self.log_initial)

            # perform expectation step
            trans_counts, emit_counts, init_counts = self.expectation(sequences)

            # convert to log probabilities
            log_trans_probs, log_emit_probs, log_init_probs = self.convert_counts(trans_counts, emit_counts, init_counts)

            # update the copy of the model with the new log probs
            for i,key in enumerate(transition_copy.keys()):
                for j,state in enumerate(self.states):
                    transition_copy[key][state] = log_trans_probs[i][j]

            for i,key in enumerate(emission_copy.keys()):
                for j,emission in enumerate(self.emissions):
                    emission_copy[key][emission] = log_emit_probs[i][j]

            for i, key in enumerate(initial_copy.keys()):
                initial_copy[key] = log_init_probs[i]


            # compare the log likelihood of our old model with our new model
            old_ll = 0.0
            for seq in sequences:
                forward_matrix, forward_prob = self.forward(seq)
                old_ll += forward_prob

            # reassignment to update model
            self.log_transition = transition_copy
            self.log_emission = emission_copy
            self.log_initial = initial_copy

            new_ll = 0.0
            for seq in sequences:
                forward_matrix, forward_prob = self.forward(seq)
                new_ll += forward_prob

    
            print(new_ll - old_ll)
                
            if abs(new_ll - old_ll) < threshold:
                print("Convergence reached! Your model has been trained")
                return self.log_transition, self.log_emission, self.log_initial

        print("Convergence criteria not met")
        return self.log_transition, self.log_emission, self.log_initial


def model_randomizer(states, emissions, seed=7):
    """
    Creates a random model
    """
    rng = np.random.default_rng(seed)

    init_probs = defaultdict()
    trans_probs = defaultdict(defaultdict)
    emit_probs = defaultdict(defaultdict)

    random_ints = rng.integers(1, 101, size=len(states))
    for i in range(len(states)):
        init_probs[states[i]] = (random_ints[i] / sum(random_ints))

    for i in range(len(states)):
        random_ints = rng.integers(1, 101, size=len(states))
        for j in range(len(states)):
            trans_probs[states[i]][states[j]] = random_ints[j] / sum(random_ints)

    for i in range(len(states)):
        random_ints = rng.integers(1, 101, size = len(emissions))
        for j in range(len(emissions)):
            emit_probs[states[i]][emissions[j]] = random_ints[j] / sum(random_ints)

    return init_probs, trans_probs, emit_probs


## Example Usage
Below is an example of how you can train the algorithm on a test sample. The test samples is the small one provided for the assignment, and are not sufficient for training or convergence, they are just meant to demonstrate usage. For sufficient training a much larger test set should be supplied, along with the model you want to use. Additionaly if you want to use a random model, just provide `model_randomizer()` with the states and emissions as lists of strings to create random probabilities for your model. An important note is that all values are in log space, this is what we've worked in throughout the HMM models, so we've kept everything in for consistency.

In [38]:

# Example observation sequences (multiple sequences for training)
obs = ["GGCACTGAA", "ATGCAATGC", "AATGCCTGA"]

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "H": 0.5,  # H = High GC content state
    "L": 0.5   # L = Low GC content state
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "H": {"H": 0.6, "L": 0.4},
    "L": {"H": 0.3, "L": 0.7}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "H": {"A": 0.2, "C": 0.3, "G": 0.3, "T": 0.2},
    "L": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

states = ["H", "L"]

emissions = ["A", "C", "G", "T"]

# Example usage with provided probabilities
print("Log-likelihood changes over 3 iterations for the example probabilities:")
bw = BaumWelch(states, emissions, init_probs, trans_probs, emit_probs)
transition_counts, emission_counts, initial_counts = bw.expectation(obs)
trans_probs, emit_probs, init_probs = bw.maximization(obs, 0.0001, 3)
print(f"Initial probabilities for our states are {np.exp(init_probs["H"])} for a high GC state and {np.exp(init_probs["L"])} for a low GC state\n")

# Example usage with randomized probabilities
print("Log-likelihood changes over 3 iterations for randomized probabilities:")
random_init, random_trans, random_emit = model_randomizer(states, emissions, seed=7)
rbw = BaumWelch(states, emissions, random_init, random_trans, random_emit)
trans_probs, emit_probs, init_probs = bw.maximization(obs, 0.0001, 3)
print(f"Initial probabilities for our states are {np.exp(init_probs["H"])} for a high GC state and {np.exp(init_probs["L"])} for a low GC state")


Log-likelihood changes over 3 iterations for the example probabilities:
0.4726615739552358
0.024394972599537823
0.02960373958170237
Convergence criteria not met
Initial probabilities for our states are 0.38224617254716453 for a high GC state and 0.6177538274528354 for a low GC state

Log-likelihood changes over 3 iterations for randomized probabilities:
0.04042110334568605
0.056804845596268194
0.0773691486297281
Convergence criteria not met
Initial probabilities for our states are 0.23827152639128144 for a high GC state and 0.7617284736087186 for a low GC state
